In [ ]:
import csv
from pathlib import Path

def prosjecna_cijena(putanja: Path) -> float:
    ukupna_cijena = 0
    ukupno_nocenja = 0

    with open(putanja, encoding="utf-8") as fajl:
        reader = csv.DictReader(fajl)

        for red in reader:
            cijena = float(red["cijena_noci"])
            nocenja = int(red["noci"])

            ukupna_cijena += cijena * nocenja
            ukupno_nocenja += nocenja

    if ukupno_nocenja == 0:
        raise ValueError("Nema podataka o noćenjima")

    return ukupna_cijena / ukupno_nocenja

In [ ]:
podaci = [
    {"gost": "Ana", "noci": 2, "cijena_noci": 100},
    {"gost": "Marko", "noci": 3, "cijena_noci": 200},
    {"gost": "Jovana", "noci": 1, "cijena_noci": 150}
]

with open("rezervacije.csv", "w", newline="", encoding="utf-8") as fajl:
    writer = csv.DictWriter(
        fajl,
        fieldnames=["gost", "noci", "cijena_noci"]
    )
    writer.writeheader()
    writer.writerows(podaci)

rezultat = prosjecna_cijena(Path("rezervacije.csv"))

print("Prosječna cijena:", rezultat)

Prosječna cijena: 158.33333333333334


In [ ]:
# Test 1
assert prosjecna_cijena(Path("rezervacije.csv")) == 160
print("Test 1 je uspješno prošao!")

Test 1 je uspješno prošao!


In [ ]:
%%writefile main.py
import csv
from pathlib import Path

def prosjecna_cijena(putanja: Path) -> float:
    ukupna_cijena = 0.0
    ukupno_nocenja = 0

    with open(putanja, encoding="utf-8", newline="") as fajl:
        reader = csv.DictReader(fajl)

        for red in reader:
            try:
                cijena = float(red["cijena_noci"])
                nocenja = int(red["noci"])
            except (ValueError, TypeError, KeyError) as greska:
                raise ValueError("Neispravni podaci u CSV fajlu") from greska

            ukupna_cijena += cijena * nocenja
            ukupno_nocenja += nocenja

    if ukupno_nocenja == 0:
        raise ValueError("CSV fajl nema validnih rezervacija")

    return ukupna_cijena / ukupno_nocenja

Writing main.py


In [ ]:
%pip install pytest

In [ ]:
%%writefile test_main.py
import pytest
from pathlib import Path
from main import prosjecna_cijena


def test_prosjecna_cijena(tmp_path):
    fajl = tmp_path / "rezervacije.csv"
    fajl.write_text(
        "gost,noci,cijena_noci\n"
        "Ana,2,100\n"
        "Marko,3,200\n"
        "Jovana,1,150\n",
        encoding="utf-8"
    )

    assert prosjecna_cijena(fajl) == pytest.approx(158.3333333333)


def test_prazan_csv(tmp_path):
    fajl = tmp_path / "prazan.csv"
    fajl.write_text(
        "gost,noci,cijena_noci\n",
        encoding="utf-8"
    )

    with pytest.raises(ValueError):
        prosjecna_cijena(fajl)


def test_neispravni_podaci(tmp_path):
    fajl = tmp_path / "greska.csv"
    fajl.write_text(
        "gost,noci,cijena_noci\n"
        "Ana,dva,100\n",
        encoding="utf-8"
    )

    with pytest.raises(ValueError):
        prosjecna_cijena(fajl)

Writing test_main.py


In [ ]:
!pytest -q

...                                                                      [100%]
3 passed in 0.02s


In [ ]:
%pip install ruff

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 26.0 MB/s  0:00:00


In [ ]:
!ruff check main.py test_main.py

I001 [*] Import block is un-sorted or un-formatted
 --> main.py:1:1
  |
1 | / import csv
2 | | from pathlib import Path
  | |________________________^
3 |
4 |   def prosjecna_cijena(putanja: Path) -> float:
  |
help: Organize imports
  |
3 |
4 +
5 | def prosjecna_cijena(putanja: Path) -> float:
  |

I001 [*] Import block is un-sorted or un-formatted
 --> test_main.py:1:1
  |
1 | / import pytest
2 | | from pathlib import Path
3 | | from main import prosjecna_cijena
  | |_________________________________^
help: Organize imports
  |
1 + from pathlib import Path
2 +
3 | import pytest
  - from pathlib import Path
4 +
5 | from main import prosjecna_cijena
  |

F401 [*] `pathlib.Path` imported but unused
 --> test_main.py:2:21
  |
1 | import pytest
2 | from pathlib import Path
  |                     ^^^^
3 | from main import prosjecna_cijena
  |
help: Remove unused import: `pathlib.Path`
  |
1 | import pytest
  - from pathlib import Path
2 | from main import prosjecna_cijena
  |

Found 3 err

In [ ]:
!ruff check main.py test_main.py --fix

Found 3 errors (3 fixed, 0 remaining).


In [ ]:
!ruff check main.py test_main.py

All checks passed!


In [ ]:
%%writefile README.md
# Zadatak 1 – Prosječna cijena rezervacije

**Autor:** Ivona Dabović
**Indeks:** 26/019
**Predmet:** Napredno programiranje

## Opis projekta

Projekat predstavlja Python funkciju koja računa ponderisanu prosječnu cijenu noćenja na osnovu podataka iz CSV fajla.

Fajl sadrži kolone:
- gost
- noci
- cijena_noci

## Korišćeni alati

- Python 3.12+
- Google Colab
- pytest
- Ruff
- ChatGPT

## Testiranje

Pokrenuti testove komandom:

    pytest -q

Rezultat: 3 passed.

Provjera kvaliteta koda:

    ruff check main.py test_main.py

Rezultat: All checks passed.

## AI prompt

Koristila sam ChatGPT uz sljedeći zahtjev:

„Implementiraj Python funkciju prosjecna_cijena koja učitava CSV fajl sa kolonama gost, noci i cijena_noci i računa ponderisanu prosječnu cijenu noćenja. Koristi standardnu biblioteku, type hints i obradu grešaka za prazan fajl i neispravne podatke. Prikaži kod, pytest testove i kratko objašnjenje.“

## Uočeni nedostaci i korekcije

- Prva verzija nije imala potpunu obradu neispravnih podataka.
- Provjerila sam ponašanje funkcije na praznom CSV fajlu.
- Ruff je prijavio neuređene importe i jedan nepotreban import. Ispravila sam ih pomoću Ruff alata.
- Testirala sam funkciju na tri slučaja: ispravni podaci, prazan fajl i neispravni podaci.

Writing README.md


In [ ]:
from google.colab import files

files.download("main.py")
files.download("test_main.py")
files.download("README.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("test_main.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%%writefile mini_projekat_1.py
import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Protocol


@dataclass
class Rezervacija:
    gost: str
    noci: int
    cijena_noci: float


class IzvorRezervacija(Protocol):
    def ucitaj(self) -> list[Rezervacija]:
        ...


class Izvjestajnik(Protocol):
    def sacuvaj(self, iznosi: list[float]) -> None:
        ...


class CsvIzvor:
    def __init__(self, putanja: Path):
        self.putanja = putanja

    def ucitaj(self) -> list[Rezervacija]:
        with open(self.putanja, encoding="utf-8") as fajl:
            reader = csv.DictReader(fajl)
            return [
                Rezervacija(
                    gost=red["gost"],
                    noci=int(red["noci"]),
                    cijena_noci=float(red["cijena_noci"]),
                )
                for red in reader
            ]


class TekstualniIzvjestaj:
    def __init__(self, putanja: Path):
        self.putanja = putanja

    def sacuvaj(self, iznosi: list[float]) -> None:
        with open(self.putanja, "w", encoding="utf-8") as fajl:
            fajl.write(f"Ukupno rezervacija: {len(iznosi)}\n")
            fajl.write(f"Ukupan iznos: {sum(iznosi):.2f} EUR\n")


def obracunaj_iznos(rezervacija: Rezervacija) -> float:
    return rezervacija.noci * rezervacija.cijena_noci


class ObradaRezervacija:
    def __init__(
        self,
        izvor: IzvorRezervacija,
        izvjestajnik: Izvjestajnik,
    ):
        self.izvor = izvor
        self.izvjestajnik = izvjestajnik

    def obradi(self) -> list[float]:
        rezervacije = self.izvor.ucitaj()
        iznosi = [
            obracunaj_iznos(rezervacija)
            for rezervacija in rezervacije
        ]
        self.izvjestajnik.sacuvaj(iznosi)
        return iznosi

Writing mini_projekat_1.py


In [ ]:
%%writefile test_mini_projekat_1.py
import pytest

from mini_projekat_1 import (
    ObradaRezervacija,
    Rezervacija,
    obracunaj_iznos,
)


class LazniIzvor:
    def __init__(self, rezervacije):
        self.rezervacije = rezervacije

    def ucitaj(self):
        return self.rezervacije


class LazniIzvjestaj:
    def __init__(self):
        self.iznosi = None

    def sacuvaj(self, iznosi):
        self.iznosi = iznosi


def test_obracun_iznosa():
    rezervacija = Rezervacija("Ana", 3, 80)
    assert obracunaj_iznos(rezervacija) == 240


def test_obrada_vise_rezervacija():
    izvor = LazniIzvor([
        Rezervacija("Ana", 3, 80),
        Rezervacija("Marko", 2, 100),
    ])
    izvjestaj = LazniIzvjestaj()

    servis = ObradaRezervacija(izvor, izvjestaj)
    rezultat = servis.obradi()

    assert rezultat == [240, 200]
    assert izvjestaj.iznosi == [240, 200]


def test_prazan_izvor():
    izvor = LazniIzvor([])
    izvjestaj = LazniIzvjestaj()

    servis = ObradaRezervacija(izvor, izvjestaj)
    rezultat = servis.obradi()

    assert rezultat == []
    assert izvjestaj.iznosi == []

Writing test_mini_projekat_1.py


In [ ]:
!pytest -q

......                                                                   [100%]
6 passed in 0.03s


In [ ]:
!ruff check mini_projekat_1.py test_mini_projekat_1.py

F401 [*] `pytest` imported but unused
 --> test_mini_projekat_1.py:1:8
  |
1 | import pytest
  |        ^^^^^^
2 |
3 | from mini_projekat_1 import (
  |
help: Remove unused import: `pytest`
  |
  - import pytest
1 |
  |

Found 1 error.
[*] 1 fixable with `--fix`.


In [ ]:
!ruff check mini_projekat_1.py test_mini_projekat_1.py --fix

Found 1 error (1 fixed, 0 remaining).


In [ ]:
!ruff check mini_projekat_1.py test_mini_projekat_1.py

All checks passed!


In [ ]:
!pytest -q

......                                                                   [100%]
6 passed in 0.02s


In [ ]:
%%writefile README.md

# Napredno programiranje

**Autor:** Ivona Dabović
**Indeks:** 26/019
**Studijski program:** Vještačka inteligencija

## Korišćeni alati

- Python 3.12+
- Google Colab
- pytest
- Ruff
- ChatGPT

---

## Zadatak 1 – Prosječna cijena rezervacije

### Opis

Implementirana je funkcija `prosjecna_cijena` koja učitava podatke iz CSV fajla i računa ponderisanu prosječnu cijenu noćenja.

CSV fajl sadrži kolone:
- gost
- noci
- cijena_noci

### Testiranje

Implementirana su tri testa:
- Ispravni podaci
- Prazan CSV fajl
- Neispravni podaci

Rezultat: 3 uspješna testa.

### AI prompt

„Implementiraj Python funkciju prosjecna_cijena koja učitava CSV fajl sa kolonama gost, noci i cijena_noci i računa ponderisanu prosječnu cijenu noćenja. Koristi standardnu biblioteku, type hints i obradu grešaka za prazan fajl i neispravne podatke. Prikaži kod, pytest testove i kratko objašnjenje.“

### Uočeni nedostaci i korekcije

- Provjerena je obrada neispravnih podataka.
- Dodata je provjera praznog CSV fajla.
- Ruff je prijavio neuređene i nepotrebne importe, koji su ispravljeni.
- Funkcija je testirana na tri različita slučaja.

---

## Mini projekat 1 – Refaktorisanje obrade rezervacija

### Opis

Cilj projekta je refaktorisanje monolitnog programa za obradu rezervacija primjenom principa SOLID.

Odgovornosti su razdvojene na posebne klase i funkcije:
- `Rezervacija` – predstavlja podatke o rezervaciji.
- `CsvIzvor` – učitava rezervacije iz CSV fajla.
- `TekstualniIzvjestaj` – čuva izvještaj.
- `obracunaj_iznos` – računa iznos rezervacije.
- `ObradaRezervacija` – upravlja procesom obrade.

Korišćeni su `Protocol` interfejsi za izvor podataka i izvještavanje.

### Primijenjeni principi

- **Single Responsibility Principle (SRP):** Svaka klasa ima jasno definisanu odgovornost.
- **Open/Closed Principle (OCP):** Moguće je dodati drugi izvor podataka bez izmjene servisa za obradu.
- **Dependency Inversion Principle (DIP):** Servis zavisi od protokola, a ne od konkretnih implementacija.

### Testiranje

Implementirana su tri testa:
- Obračun iznosa jedne rezervacije.
- Obrada više rezervacija.
- Obrada praznog izvora podataka.

Za testiranje su korišćeni lažni izvori podataka i lažni izvještajnik.

Rezultat: 6 uspješnih testova ukupno za oba projekta.

### AI prompt

„Ponašaj se kao iskusan softverski arhitekta. Analiziraj dati Python kod za obradu rezervacija. Identifikuj povrede SOLID principa, objasni probleme i predloži refaktorisanje razdvajanjem odgovornosti. Uvedi Protocol za izvor podataka i izvještavanje. Predloži najmanje tri pytest testa koristeći lažni izvor podataka. Izbjegavaj nepotrebne apstrakcije.“

### Uočeni nedostaci i korekcije

- Početni monolitni kod imao je više odgovornosti u jednoj klasi.
- Razdvojene su odgovornosti za učitavanje, obračun i izvještavanje.
- Uvedeni su protokoli radi lakše zamjene implementacija.
- Dodati su testovi sa lažnim izvorom i izvještajnikom.
- Ruff je korišćen za provjeru i ispravljanje kvaliteta koda.

---

## Pokretanje testova

```bash
pytest -q

Overwriting README.md


In [ ]:
!ruff check .

All checks passed!


In [ ]:
!pytest -q

......                                                                   [100%]
6 passed in 0.02s


In [ ]:
import zipfile

fajlovi = [
    "mini_projekat_1.py",
    "test_mini_projekat_1.py",
    "README.md"
]

with zipfile.ZipFile("mini_projekat_1_fajlovi.zip", "w") as zipf:
    for fajl in fajlovi:
        zipf.write(fajl)

from google.colab import files
files.download("mini_projekat_1_fajlovi.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from dataclasses import dataclass


@dataclass
class Rezervacija:
    gost: str
    noci: int
    cijena_noci: float

    def __post_init__(self):
        if not self.gost.strip():
            raise ValueError("Ime gosta ne smije biti prazno")

        if self.noci <= 0:
            raise ValueError("Broj nocenja mora biti veci od nule")

        if self.cijena_noci < 0:
            raise ValueError("Cijena nocenja ne moze biti negativna")

In [ ]:
import csv
from pathlib import Path


class CsvIzvor:
    def __init__(self, putanja: Path):
        self.putanja = putanja

    def ucitaj(self) -> list[Rezervacija]:
        with open(
            self.putanja,
            encoding="utf-8",
            newline=""
        ) as fajl:
            reader = csv.DictReader(fajl)

            potrebne_kolone = {"gost", "noci", "cijena_noci"}

            if not reader.fieldnames or not potrebne_kolone.issubset(reader.fieldnames):
                raise ValueError("CSV fajl nema sve potrebne kolone")

            rezervacije = []

            for red in reader:
                try:
                    rezervacija = Rezervacija(
                        gost=red["gost"],
                        noci=int(red["noci"]),
                        cijena_noci=float(red["cijena_noci"])
                    )
                    rezervacije.append(rezervacija)

                except (ValueError, TypeError) as greska:
                    raise ValueError(
                        f"Neispravni podaci u CSV fajlu: {red}"
                    ) from greska

            return rezervacije

In [ ]:
def obracunaj_iznos(rezervacija: Rezervacija) -> float:
    if rezervacija.noci <= 0:
        raise ValueError("Broj nocenja mora biti veci od nule")

    if rezervacija.cijena_noci < 0:
        raise ValueError("Cijena nocenja ne moze biti negativna")

    return rezervacija.noci * rezervacija.cijena_noci

In [ ]:
class TekstualniIzvjestaj:
    def __init__(self, putanja: Path):
        self.putanja = putanja

    def sacuvaj(self, iznosi: list[float]) -> None:
        with open(self.putanja, "w", encoding="utf-8") as fajl:
            fajl.write(f"Ukupno rezervacija: {len(iznosi)}\n")
            fajl.write(f"Ukupan iznos: {sum(iznosi):.2f} EUR\n")

In [ ]:
class ObradaRezervacija:
    def __init__(
        self,
        izvor: IzvorRezervacija,
        izvjestajnik: Izvjestajnik,
    ):
        self.izvor = izvor
        self.izvjestajnik = izvjestajnik

    def obradi(self) -> list[float]:
        rezervacije = self.izvor.ucitaj()

        iznosi = [
            obracunaj_iznos(rezervacija)
            for rezervacija in rezervacije
        ]

        self.izvjestajnik.sacuvaj(iznosi)

        return iznosi

In [ ]:
from typing import Protocol


class IzvorRezervacija(Protocol):
    def ucitaj(self) -> list[Rezervacija]:
        ...


class Izvjestajnik(Protocol):
    def sacuvaj(self, iznosi: list[float]) -> None:
        ...

In [ ]:
def test_prazno_ime_gosta():
    with pytest.raises(ValueError):
        Rezervacija("", 2, 100)


def test_ispravna_rezervacija():
    rezervacija = Rezervacija("Ana", 3, 80)

    assert rezervacija.gost == "Ana"
    assert rezervacija.noci == 3
    assert rezervacija.cijena_noci == 80


def test_obracun_iznosa():
    rezervacija = Rezervacija("Ana", 3, 80)

    assert obracunaj_iznos(rezervacija) == 240

In [ ]:
test_prazno_ime_gosta()
test_ispravna_rezervacija()
test_obracun_iznosa()
test_rezervacija_sa_nula_nocenja()
test_rezervacija_sa_negativnom_cijenom()
test_csv_bez_potrebnih_kolona()

print("Svi dodatni testovi su uspjesno prosli!")

Svi dodatni testovi su uspjesno prosli!


In [ ]:
import pytest

In [ ]:
def test_rezervacija_sa_nula_nocenja():
    with pytest.raises(ValueError):
        Rezervacija("Ana", 0, 80)


def test_rezervacija_sa_negativnom_cijenom():
    with pytest.raises(ValueError):
        Rezervacija("Ana", 3, -80)


def test_csv_bez_potrebnih_kolona(tmp_path):
    fajl = tmp_path / "pogresan.csv"

    fajl.write_text(
        "gost,noci\nAna,3\n",
        encoding="utf-8"
    )

    izvor = CsvIzvor(fajl)

    with pytest.raises(ValueError):
        izvor.ucitaj()

In [ ]:
from pathlib import Path
import tempfile


def test_csv_bez_potrebnih_kolona():
    with tempfile.TemporaryDirectory() as folder:
        fajl = Path(folder) / "pogresan.csv"

        fajl.write_text(
            "gost,noci\nAna,3\n",
            encoding="utf-8"
        )

        izvor = CsvIzvor(fajl)

        with pytest.raises(ValueError):
            izvor.ucitaj()

In [ ]:
rezervacije = [
    Rezervacija("Ana", 3, 80),
    Rezervacija("Marko", 2, 100),
]

izvor = LazniIzvor(rezervacije)
izvjestaj = LazniIzvjestaj()

servis = ObradaRezervacija(izvor, izvjestaj)

rezultat = servis.obradi()

assert rezultat == [240, 200]
assert izvjestaj.iznosi == [240, 200]

print("Obrada rezervacija uspjesna!")
print("Iznosi:", rezultat)
print("Ukupno:", sum(rezultat), "EUR")

Obrada rezervacija uspjesna!
Iznosi: [240, 200]
Ukupno: 440 EUR


In [ ]:
class LazniIzvor:
    def __init__(self, rezervacije):
        self.rezervacije = rezervacije

    def ucitaj(self):
        return self.rezervacije


class LazniIzvjestaj:
    def __init__(self):
        self.iznosi = None

    def sacuvaj(self, iznosi):
        self.iznosi = iznosi

In [ ]:
%%writefile mini_projekat_1.py

import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Protocol


@dataclass
class Rezervacija:
    gost: str
    noci: int
    cijena_noci: float

    def __post_init__(self):
        if not self.gost.strip():
            raise ValueError("Ime gosta ne smije biti prazno")
        if self.noci <= 0:
            raise ValueError("Broj nocenja mora biti veci od nule")
        if self.cijena_noci < 0:
            raise ValueError("Cijena nocenja ne moze biti negativna")


class IzvorRezervacija(Protocol):
    def ucitaj(self) -> list[Rezervacija]:
        ...


class Izvjestajnik(Protocol):
    def sacuvaj(self, iznosi: list[float]) -> None:
        ...


class CsvIzvor:
    def __init__(self, putanja: Path):
        self.putanja = putanja

    def ucitaj(self) -> list[Rezervacija]:
        with open(self.putanja, encoding="utf-8", newline="") as fajl:
            reader = csv.DictReader(fajl)

            potrebne_kolone = {"gost", "noci", "cijena_noci"}

            if not reader.fieldnames or not potrebne_kolone.issubset(reader.fieldnames):
                raise ValueError("CSV fajl nema sve potrebne kolone")

            rezervacije = []

            for red in reader:
                try:
                    rezervacija = Rezervacija(
                        gost=red["gost"],
                        noci=int(red["noci"]),
                        cijena_noci=float(red["cijena_noci"])
                    )
                    rezervacije.append(rezervacija)
                except (ValueError, TypeError) as greska:
                    raise ValueError(
                        f"Neispravni podaci u CSV fajlu: {red}"
                    ) from greska

            return rezervacije


class TekstualniIzvjestaj:
    def __init__(self, putanja: Path):
        self.putanja = putanja

    def sacuvaj(self, iznosi: list[float]) -> None:
        with open(self.putanja, "w", encoding="utf-8") as fajl:
            fajl.write(f"Ukupno rezervacija: {len(iznosi)}\n")
            fajl.write(f"Ukupan iznos: {sum(iznosi):.2f} EUR\n")


def obracunaj_iznos(rezervacija: Rezervacija) -> float:
    return rezervacija.noci * rezervacija.cijena_noci


class ObradaRezervacija:
    def __init__(
        self,
        izvor: IzvorRezervacija,
        izvjestajnik: Izvjestajnik,
    ):
        self.izvor = izvor
        self.izvjestajnik = izvjestajnik

    def obradi(self) -> list[float]:
        rezervacije = self.izvor.ucitaj()
        iznosi = [
            obracunaj_iznos(rezervacija)
            for rezervacija in rezervacije
        ]
        self.izvjestajnik.sacuvaj(iznosi)
        return iznosi

Writing mini_projekat_1.py


In [ ]:
%%writefile test_mini_projekat_1.py

import pytest

from pathlib import Path
from mini_projekat_1 import (
    Rezervacija,
    CsvIzvor,
    ObradaRezervacija,
    obracunaj_iznos,
)


class LazniIzvor:
    def __init__(self, rezervacije):
        self.rezervacije = rezervacije

    def ucitaj(self):
        return self.rezervacije


class LazniIzvjestaj:
    def __init__(self):
        self.iznosi = None

    def sacuvaj(self, iznosi):
        self.iznosi = iznosi


def test_obracun_iznosa():
    rezervacija = Rezervacija("Ana", 3, 80)
    assert obracunaj_iznos(rezervacija) == 240


def test_obrada_vise_rezervacija():
    rezervacije = [
        Rezervacija("Ana", 3, 80),
        Rezervacija("Marko", 2, 100),
    ]

    izvor = LazniIzvor(rezervacije)
    izvjestaj = LazniIzvjestaj()

    servis = ObradaRezervacija(izvor, izvjestaj)

    assert servis.obradi() == [240, 200]
    assert izvjestaj.iznosi == [240, 200]


def test_prazan_izvor():
    izvor = LazniIzvor([])
    izvjestaj = LazniIzvjestaj()

    servis = ObradaRezervacija(izvor, izvjestaj)

    assert servis.obradi() == []
    assert izvjestaj.iznosi == []


def test_neispravni_csv_podaci(tmp_path):
    fajl = tmp_path / "greska.csv"

    fajl.write_text(
        "gost,noci,cijena_noci\nAna,dva,100\n",
        encoding="utf-8"
    )

    with pytest.raises(ValueError):
        CsvIzvor(fajl).ucitaj()


def test_csv_bez_potrebnih_kolona(tmp_path):
    fajl = tmp_path / "pogresan.csv"

    fajl.write_text(
        "gost,noci\nAna,3\n",
        encoding="utf-8"
    )

    with pytest.raises(ValueError):
        CsvIzvor(fajl).ucitaj()


def test_rezervacija_sa_negativnom_cijenom():
    with pytest.raises(ValueError):
        Rezervacija("Ana", 3, -80)


def test_rezervacija_sa_nula_nocenja():
    with pytest.raises(ValueError):
        Rezervacija("Ana", 0, 80)%%writefile test_mini_projekat_1.py

import pytest

from pathlib import Path
from mini_projekat_1 import (
    Rezervacija,
    CsvIzvor,
    ObradaRezervacija,
    obracunaj_iznos,
)


class LazniIzvor:
    def __init__(self, rezervacije):
        self.rezervacije = rezervacije

    def ucitaj(self):
        return self.rezervacije


class LazniIzvjestaj:
    def __init__(self):
        self.iznosi = None

    def sacuvaj(self, iznosi):
        self.iznosi = iznosi


def test_obracun_iznosa():
    rezervacija = Rezervacija("Ana", 3, 80)
    assert obracunaj_iznos(rezervacija) == 240


def test_obrada_vise_rezervacija():
    rezervacije = [
        Rezervacija("Ana", 3, 80),
        Rezervacija("Marko", 2, 100),
    ]

    izvor = LazniIzvor(rezervacije)
    izvjestaj = LazniIzvjestaj()

    servis = ObradaRezervacija(izvor, izvjestaj)

    assert servis.obradi() == [240, 200]
    assert izvjestaj.iznosi == [240, 200]


def test_prazan_izvor():
    izvor = LazniIzvor([])
    izvjestaj = LazniIzvjestaj()

    servis = ObradaRezervacija(izvor, izvjestaj)

    assert servis.obradi() == []
    assert izvjestaj.iznosi == []


def test_neispravni_csv_podaci(tmp_path):
    fajl = tmp_path / "greska.csv"

    fajl.write_text(
        "gost,noci,cijena_noci\nAna,dva,100\n",
        encoding="utf-8"
    )

    with pytest.raises(ValueError):
        CsvIzvor(fajl).ucitaj()


def test_csv_bez_potrebnih_kolona(tmp_path):
    fajl = tmp_path / "pogresan.csv"

    fajl.write_text(
        "gost,noci\nAna,3\n",
        encoding="utf-8"
    )

    with pytest.raises(ValueError):
        CsvIzvor(fajl).ucitaj()


def test_rezervacija_sa_negativnom_cijenom():
    with pytest.raises(ValueError):
        Rezervacija("Ana", 3, -80)


def test_rezervacija_sa_nula_nocenja():
    with pytest.raises(ValueError):
        Rezervacija("Ana", 0, 80)


Overwriting test_mini_projekat_1.py


In [ ]:
!pytest -q test_mini_projekat_1.py


==================================== ERRORS ====================================
___________________ ERROR collecting test_mini_projekat_1.py ___________________
/usr/local/lib/python3.13/dist-packages/_pytest/python.py:498: in importtestmodule
    mod = import_path(
/usr/local/lib/python3.13/dist-packages/_pytest/pathlib.py:587: in import_path
    importlib.import_module(module_name)
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
<frozen importlib._bootstrap>:1404: in _gcd_import
    ???
<frozen importlib._bootstrap>:1369: in _find_and_load
    ???
<frozen importlib._bootstrap>:1340: in _find_and_load_unlocked
    ???
<frozen importlib._bootstrap>:935: in _load_unlocked
    ???
/usr/local/lib/python3.13/dist-packages/_pytest/assertion/rewrite.py:177: in exec_module
    source_stat, co = _rewrite_test(fn, self.config)
                      ^^^

In [ ]:
!tail -n 15 test_mini_projekat_1.py

        encoding="utf-8"
    )

    with pytest.raises(ValueError):
        CsvIzvor(fajl).ucitaj()


def test_rezervacija_sa_negativnom_cijenom():
    with pytest.raises(ValueError):
        Rezervacija("Ana", 3, -80)


def test_rezervacija_sa_nula_nocenja():
    with pytest.raises(ValueError):
        Rezervacija("Ana", 0, 80)


In [ ]:
!pytest -q test_mini_projekat_1.py

.......                                                                  [100%]
7 passed in 0.02s


In [ ]:
!nl -ba test_mini_projekat_1.py | sed -n '85,95p'
!python -m py_compile test_mini_projekat_1.py

    85	        Rezervacija("Ana", 3, -80)
    86	
    87	
    88	def test_rezervacija_sa_nula_nocenja():
    89	    with pytest.raises(ValueError):
    90	        Rezervacija("Ana", 0, 80)%%writefile test_mini_projekat_1.py
    91	
    92	import pytest
    93	
    94	from pathlib import Path
    95	from mini_projekat_1 import (
  File "test_mini_projekat_1.py", line 90
    Rezervacija("Ana", 0, 80)%%writefile test_mini_projekat_1.py
                              ^
SyntaxError: invalid syntax


In [ ]:
%%writefile test_mini_projekat_1.py
import pytest

from mini_projekat_1 import (
    Rezervacija,
    CsvIzvor,
    ObradaRezervacija,
    obracunaj_iznos,
)


class LazniIzvor:
    def __init__(self, rezervacije):
        self.rezervacije = rezervacije

    def ucitaj(self):
        return self.rezervacije


class LazniIzvjestaj:
    def __init__(self):
        self.iznosi = None

    def sacuvaj(self, iznosi):
        self.iznosi = iznosi


def test_obracun_iznosa():
    rezervacija = Rezervacija("Ana", 3, 80)
    assert obracunaj_iznos(rezervacija) == 240


def test_obrada_vise_rezervacija():
    rezervacije = [
        Rezervacija("Ana", 3, 80),
        Rezervacija("Marko", 2, 100),
    ]

    izvor = LazniIzvor(rezervacije)
    izvjestaj = LazniIzvjestaj()
    servis = ObradaRezervacija(izvor, izvjestaj)

    assert servis.obradi() == [240, 200]
    assert izvjestaj.iznosi == [240, 200]


def test_prazan_izvor():
    izvor = LazniIzvor([])
    izvjestaj = LazniIzvjestaj()
    servis = ObradaRezervacija(izvor, izvjestaj)

    assert servis.obradi() == []
    assert izvjestaj.iznosi == []


def test_neispravni_csv_podaci(tmp_path):
    fajl = tmp_path / "greska.csv"
    fajl.write_text(
        "gost,noci,cijena_noci\nAna,dva,100\n",
        encoding="utf-8"
    )

    with pytest.raises(ValueError):
        CsvIzvor(fajl).ucitaj()


def test_csv_bez_potrebnih_kolona(tmp_path):
    fajl = tmp_path / "pogresan.csv"
    fajl.write_text(
        "gost,noci\nAna,3\n",
        encoding="utf-8"
    )

    with pytest.raises(ValueError):
        CsvIzvor(fajl).ucitaj()


def test_rezervacija_sa_negativnom_cijenom():
    with pytest.raises(ValueError):
        Rezervacija("Ana", 3, -80)


def test_rezervacija_sa_nula_nocenja():
    with pytest.raises(ValueError):
        Rezervacija("Ana", 0, 80)

Overwriting test_mini_projekat_1.py


In [ ]:
!pytest -q test_mini_projekat_1.py

.......                                                                  [100%]
7 passed in 0.02s


In [ ]:
!ruff check mini_projekat_1.py test_mini_projekat_1.py


/bin/bash: line 1: ruff: command not found


In [ ]:
!pip install ruff

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 6.1 MB/s  0:00:01


In [ ]:
!ruff check mini_projekat_1.py test_mini_projekat_1.py

All checks passed!


In [ ]:
!ruff check --fix test_mini_projekat_1.py

Found 1 error (1 fixed, 0 remaining).


In [ ]:
!pytest -q

.......                                                                  [100%]
7 passed in 0.02s


In [ ]:

%%writefile README.md
# Mini projekat 1 – SOLID refaktorisanje obrade rezervacija

## Opis projekta

Cilj projekta je refaktorisanje sistema za obradu rezervacija primjenom SOLID principa. Sistem učitava podatke o rezervacijama iz CSV fajla, obračunava ukupne iznose i čuva izvještaj.

## Funkcionalnosti

- Učitavanje rezervacija iz CSV fajla.
- Provjera ispravnosti podataka.
- Validacija imena gosta, broja noćenja i cijene.
- Obračun ukupnog iznosa rezervacije.
- Generisanje tekstualnog izvještaja.
- Obrada praznog izvora podataka.

## Struktura projekta

- `mini_projekat_1.py` – glavna implementacija.
- `test_mini_projekat_1.py` – automatski testovi.
- `README.md` – dokumentacija projekta.

## Primjena SOLID principa

**Single Responsibility Principle (SRP)**

Svaka klasa ima jasno definisanu odgovornost:
- `CsvIzvor` učitava podatke iz CSV fajla.
- `TekstualniIzvjestaj` čuva izvještaj.
- `ObradaRezervacija` upravlja procesom obrade.
- `Rezervacija` predstavlja podatke o jednoj rezervaciji.

**Dependency Inversion Principle (DIP)**

Klasa `ObradaRezervacija` zavisi od apstrakcija definisanih pomoću `Protocol`, a ne od konkretnih implementacija.

Protokoli:
- `IzvorRezervacija`
- `Izvjestajnik`

Na ovaj način moguće je koristiti različite izvore podataka i načine čuvanja izvještaja bez izmjene glavne klase za obradu.

## Pokretanje testova

Za instalaciju potrebnih alata:

```bash
pip install pytest ruff
```

Pokretanje testova:

```bash
pytest -q
```

Provjera kvaliteta koda:

```bash
ruff check mini_projekat_1.py test_mini_projekat_1.py
```

## Testiranje

Projekat sadrži 7 automatskih testova koji provjeravaju:
- obračun iznosa rezervacije;
- obradu više rezervacija;
- obradu praznog izvora;
- neispravne podatke u CSV fajlu;
- nedostajuće CSV kolone;
- negativnu cijenu;
- nula noćenja.

Rezultat testiranja: **7 passed**.

## Korišćenje AI alata

AI alat je korišćen kao pomoć pri razumijevanju SOLID principa, refaktorisanju koda, organizaciji testova i otklanjanju grešaka.

Predložena rješenja su provjerena kroz izvršavanje programa, automatske testove i Ruff alat.

## Zaključak

Refaktorisanjem je postignuta bolja organizacija koda, jasnije razdvajanje odgovornosti i lakše testiranje. Zahvaljujući protokolima, sistem je fleksibilniji i jednostavniji za proširivanje.


Writing README.md


In [ ]:
from google.colab import files

files.download("mini_projekat_1.py")
files.download("test_mini_projekat_1.py")
files.download("README.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>